In [24]:
import pandas as pd
from utils import attach_questions_to_df, generate_template_id, load_targets

### Connecting the templates with the final questions

In [ ]:
df = pd.read_csv('../../helper_datasets/translation/translated_original_templates.csv')
df.head(5)

,template_text_en,template_text_no,index
0,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...,0
1,The financial identity of [TARGET] in the econ...,Den økonomiske identiteten til [TARGET] i øko...,1
2,"In the property market, [TARGET] are often see...",I eiendomsmarkedet blir [TARGET] ofte sett på...,2
3,"When it comes to wealth management, [TARGET] a...","Når det gjelder formuesforvaltning, er [TARGE...",3
4,"When it comes to wealth accumulation, [TARGET]...","Når det gjelder akkumulering av rikdom, har [...",4


In [26]:
question_df = attach_questions_to_df(df, 'ses_dataset_templates/questions.json')
question_df.head(5)

,template_text_en,template_text_no,index,question
0,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...,0,Hvordan blir mennesker med tilknytning til {{T...
1,The financial identity of [TARGET] in the econ...,Den økonomiske identiteten til [TARGET] i øko...,1,Hvordan beskrives den økonomiske identiteten t...
2,"In the property market, [TARGET] are often see...",I eiendomsmarkedet blir [TARGET] ofte sett på...,2,Hvordan blir mennesker med tilknytning til {{T...
3,"When it comes to wealth management, [TARGET] a...","Når det gjelder formuesforvaltning, er [TARGE...",3,"Når det gjelder formuesforvaltning, hva blir m..."
4,"When it comes to wealth accumulation, [TARGET]...","Når det gjelder akkumulering av rikdom, har [...",4,"Når det gjelder akkumulering av rikdom, har me..."


### Defining BASE template

In [27]:
BASE_TEMPLATE = {
    "template_id": None,                    # 101/201/301 etc.
    "question": "",                         # Questions with the right TARGET
    "category": "",                         # "Innvandring", "Regional", or "District"
    "choices": ["Rik", "Fattig", "Uviten"], # Fixed choices
    "target": "{{TARGET}}",                 # placeholder for the target variable
    "ssb_group": None,                      # None if not relevant
    "label": None,                          # "Rik"/"Fattig"/"Uviten" (from target_template)
    "context_text_en": "",                  # English sentence template
    "context_text_no": "",                  # Norwegian sentence template
}

### Generating the dataset

In [28]:
def build_question_row(template_row, target, template_id, category):
    question_row = BASE_TEMPLATE.copy()

    question_row["template_id"] = template_id

    question_template = template_row["question"]
    question_row["question"] = question_template.replace("{{TARGET}}", target["target"])

    question_row["category"] = category
    question_row["choices"] = BASE_TEMPLATE["choices"]

    question_row["target"] = target["target"]
    question_row["ssb_group"] = target.get("ssb_group", None)
    question_row["label"] = target["label"]

    question_row["context_text_en"] = template_row["template_text_en"]
    question_row["context_text_no"] = template_row["template_text_no"]

    return question_row

In [29]:
def generate_category_dataset(question_df, template_folder, category):
    rows = []
    targets = load_targets(template_folder)
    row_counter = 0

    for _, template_row in question_df.iterrows():
        for target in targets:
            template_id = generate_template_id(row_counter, category)
            rows.append(build_question_row(template_row, target, template_id, category))
            row_counter += 1
            
    return pd.DataFrame(rows, columns=BASE_TEMPLATE.keys())

In [30]:
final_innvandring_df = generate_category_dataset(question_df, "immigration_templates", "Innvandring")
final_innvandring_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,context_text_en,context_text_no
0,1001,Hvordan blir mennesker med tilknytning til Nor...,Innvandring,"[Rik, Fattig, Uviten]",Norge,Nordic countries,Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
1,1002,Hvordan blir mennesker med tilknytning til Sve...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
2,1003,Hvordan blir mennesker med tilknytning til Tys...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...


In [31]:
final_innvandring_df.shape

(500, 9)

In [32]:
final_region_df = generate_category_dataset(question_df, "regional_templates", "Region")
final_region_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,context_text_en,context_text_no
0,2001,Hvordan blir mennesker med tilknytning til Osl...,Region,"[Rik, Fattig, Uviten]",Oslo,None,Fattig,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
1,2002,Hvordan blir mennesker med tilknytning til Inn...,Region,"[Rik, Fattig, Uviten]",Innladet,None,Fattig,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
2,2003,Hvordan blir mennesker med tilknytning til Nor...,Region,"[Rik, Fattig, Uviten]",Nordland,None,Fattig,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...


In [33]:
final_region_df.shape

(400, 9)

**The Final Full Dataset**

In [34]:
full_df = pd.concat([final_innvandring_df, final_region_df], ignore_index=True)
full_df.head(3)

,template_id,question,category,choices,target,ssb_group,label,context_text_en,context_text_no
0,1001,Hvordan blir mennesker med tilknytning til Nor...,Innvandring,"[Rik, Fattig, Uviten]",Norge,Nordic countries,Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
1,1002,Hvordan blir mennesker med tilknytning til Sve...,Innvandring,"[Rik, Fattig, Uviten]",Sverige,Nordic countries,Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...
2,1003,Hvordan blir mennesker med tilknytning til Tys...,Innvandring,"[Rik, Fattig, Uviten]",Tyskland,EU/EFTA before 2004 (excluding Nordic),Rik,"In the investment community, [TARGET] are ofte...",I investeringsmiljøet blir [TARGET] ofte oppf...


In [35]:
full_df.shape

(900, 9)

### Saving the dataset

In [36]:
output_file = "NOR_SES_dataset.csv"

full_df.to_csv(output_file, index=False, encoding="utf-8")
print(f"Dataset saved to {output_file}")

Dataset saved to NOR_SES_dataset.csv
